In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("shawon10/ckplus")

print("Path to dataset files:", path)

100%|██████████| 3.63M/3.63M [00:01<00:00, 3.01MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/shawon10/ckplus/versions/1


In [3]:
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from transformers import InstructBlipProcessor, InstructBlipForConditionalGeneration
from transformers import AutoProcessor, LlavaForConditionalGeneration
from PIL import Image
from tqdm import tqdm
from transformers import BitsAndBytesConfig
import os
import numpy as np
from sklearn.metrics import accuracy_score, classification_report

# ==========================================
# 0. 配置 (Configuration)
# ==========================================
# 你的 CK+ 路径
DATA_DIR = '/root/.cache/kagglehub/datasets/shawon10/ckplus/versions/1' 
BATCH_SIZE = 1 # 大模型显存占用大，建议 Batch Size = 1 逐张推理
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 想要评测的模型: 'instructblip' 或 'llava'
MODEL_TYPE = 'llava'  # 这里修改你想跑的模型

# CK+ 的类别映射 (根据你的数据集文件夹名字修改)
# 确保这里的 Keys 全部小写，因为我们会把模型输出转小写匹配
EMOTION_MAP = {
    'anger': 0,
    'contempt': 1,
    'disgust': 2,
    'fear': 3,
    'happy': 4,
    'sadness': 5,
    'surprise': 6
}
# 如果你的文件夹名字是 '0', '1'... 请修改这里，或者让 ImageFolder 自动生成

print(f"🔥 Starting Zero-shot Evaluation for {MODEL_TYPE} on CK+...")

# ==========================================
# 1. 数据加载 (简单的 ImageFolder)
# ==========================================
def get_ckplus_loader(data_dir):
    # LVLM 不需要复杂的 transform，只需要 Resize 这一步通常 Processor 会做
    # 但为了保险，我们转成 Tensor 前不做 normalize，保留原始 PIL Image
    # 因为 HF 的 Processor 接收 PIL
    dataset = datasets.ImageFolder(data_dir)
    
    # 获取类别索引
    class_to_idx = dataset.class_to_idx
    idx_to_class = {v: k.lower() for k, v in class_to_idx.items()}
    print(f"✅ Classes found: {idx_to_class}")
    
    # 我们这里不需要 DataLoader 的 collate_fn，因为我们要喂 PIL 图片给模型
    # 所以我们手动写个简单的循环，或者用一个返回 (path, label) 的 Dataset
    return dataset, idx_to_class

# ==========================================
# 2. 模型加载器
# ==========================================
def load_model(model_type):
    # 1. 定义 4-bit 量化配置 (这是现在最稳妥的写法)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )

    if model_type == 'instructblip':
        print("⏳ Loading InstructBLIP-7B (Vicuna)...")
        processor = InstructBlipProcessor.from_pretrained("Salesforce/instructblip-vicuna-7b")
        model = InstructBlipForConditionalGeneration.from_pretrained(
            "Salesforce/instructblip-vicuna-7b",
            quantization_config=bnb_config,  # <--- 改用 quantization_config
            device_map="auto"
        )
    elif model_type == 'llava':
        print("⏳ Loading LLaVA-1.5-7B...")
        processor = AutoProcessor.from_pretrained("llava-hf/llava-1.5-7b-hf")
        model = LlavaForConditionalGeneration.from_pretrained(
            "llava-hf/llava-1.5-7b-hf",
            quantization_config=bnb_config,  # <--- 改用 quantization_config
            device_map="auto"
        )
    else:
        raise ValueError("Unknown model type")
    
    return model, processor

# ==========================================
# 3. 核心：构造 Prompt 和解析输出
# ==========================================
def get_prompt(model_type, emotion_list_str):
    if model_type == 'instructblip':
        # InstructBLIP 喜欢简短的指令
        return f"Question: What is the emotion of the person in the image? Choose from {emotion_list_str}. Answer:"
    elif model_type == 'llava':
        # LLaVA 标准格式
        return f"USER: <image>\nWhat is the facial expression of this person? Output only one word from the following list: {emotion_list_str}.\nASSISTANT:"

def parse_output(output_text, emotion_map):
    """
    非常关键：大模型可能会输出 "I think it is happy"，我们需要提取核心词 "happy"
    """
    output_text = output_text.lower().strip()
    
    # 移除标点
    import string
    output_text = output_text.translate(str.maketrans('', '', string.punctuation))
    
    # 1. 精确匹配
    for emo in emotion_map.keys():
        # 检查 emo 是否作为独立的词出现
        if emo in output_text.split():
            return emotion_map[emo]
    
    # 2. 模糊匹配 (如果没匹配到，看包含关系)
    for emo in emotion_map.keys():
        if emo in output_text:
            return emotion_map[emo]
            
    return -1 # 解析失败 (Hallucination or formatting error)

# ==========================================
# 4. 推理循环
# ==========================================
def evaluate():
    dataset, idx_to_class_map = get_ckplus_loader(DATA_DIR)
    model, processor = load_model(MODEL_TYPE)
    
    # 构造类别字符串 "anger, contempt, ..."
    # 假设 idx_to_class_map 的 value 是 'anger', 'happy' 等
    # 如果数据集文件夹名是 '0', '1', 你需要手动建立一个 int->str 的映射
    
    # 这里的 idx_to_class_map 是 dataset 自动生成的，如果文件夹名就是 emotion name
    emotion_names = list(EMOTION_MAP.keys())
    emotion_list_str = ", ".join(emotion_names)
    
    y_true = []
    y_pred = []
    
    print("🚀 Start Inference...")
    
    # 遍历数据集
    for i in tqdm(range(len(dataset))):
        image, label_idx = dataset[i] # image is PIL, label_idx is int
        
        # 转换标签 (dataset 的 label_idx 可能和 EMOTION_MAP 不对应，需要校准)
        # 假设 dataset.classes 是 ['anger', 'contempt'...] 且按字母排序
        # 为了稳妥，我们用文件夹名反查
        folder_name = dataset.classes[label_idx].lower()
        
        # 如果文件夹名含有 emotion 关键词 (例如 "1_anger")
        true_label_mapped = -1
        for key in EMOTION_MAP:
            if key in folder_name:
                true_label_mapped = EMOTION_MAP[key]
                break
        
        if true_label_mapped == -1:
            continue # 跳过未定义的类别
            
        y_true.append(true_label_mapped)
        
        # 准备输入
        prompt = get_prompt(MODEL_TYPE, emotion_list_str)
        
        if MODEL_TYPE == 'instructblip':
            inputs = processor(images=image, text=prompt, return_tensors="pt").to(DEVICE)
        elif MODEL_TYPE == 'llava':
            inputs = processor(images=image, text=prompt, return_tensors="pt").to(DEVICE)

        # 生成
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=20,
                do_sample=False, # Greedy decoding (deterministic)
                min_length=1
            )
            
        output_text = processor.batch_decode(output_ids, skip_special_tokens=True)[0].strip()
        
        # 解析
        pred_label = parse_output(output_text, EMOTION_MAP)
        
        # 如果解析失败，默认给个错的或者随机，这里我们给 -1
        y_pred.append(pred_label)
        
        # 打印前几个看看效果
        if i < 5:
            print(f"\nExample {i}:")
            print(f"Prompt: {prompt}")
            print(f"Raw Output: {output_text}")
            print(f"Parsed Pred: {pred_label}, True: {true_label_mapped}")

    # 计算指标
    # 过滤掉解析失败的 (-1)
    valid_indices = [k for k, x in enumerate(y_pred) if x != -1]
    y_true_valid = [y_true[k] for k in valid_indices]
    y_pred_valid = [y_pred[k] for k in valid_indices]
    
    acc = accuracy_score(y_true_valid, y_pred_valid)
    print(f"\n\n🏁 Final Results for {MODEL_TYPE} on CK+:")
    print(f"Parsing Success Rate: {len(valid_indices)/len(y_pred):.2%}")
    print(f"Accuracy: {acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true_valid, y_pred_valid, target_names=emotion_names))

if __name__ == '__main__':
    evaluate()

🔥 Starting Zero-shot Evaluation for llava on CK+...
✅ Classes found: {0: 'ck+48', 1: 'ck'}
⏳ Loading LLaVA-1.5-7B...


ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

In [ ]:
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from transformers import InstructBlipProcessor, InstructBlipForConditionalGeneration
from transformers import AutoProcessor, LlavaForConditionalGeneration
from PIL import Image
from tqdm import tqdm
import os
import numpy as np
from sklearn.metrics import accuracy_score, classification_report

# ==========================================
# 0. 配置 (Configuration)
# ==========================================
# 你的 CK+ 路径
DATA_DIR = '/kaggle/input/ckplus/CK+48' 
BATCH_SIZE = 1 # 大模型显存占用大，建议 Batch Size = 1 逐张推理
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 想要评测的模型: 'instructblip' 或 'llava'
MODEL_TYPE = 'instructblip'  # 这里修改你想跑的模型

# CK+ 的类别映射 (根据你的数据集文件夹名字修改)
# 确保这里的 Keys 全部小写，因为我们会把模型输出转小写匹配
EMOTION_MAP = {
    'anger': 0,
    'contempt': 1,
    'disgust': 2,
    'fear': 3,
    'happy': 4,
    'sadness': 5,
    'surprise': 6
}
# 如果你的文件夹名字是 '0', '1'... 请修改这里，或者让 ImageFolder 自动生成

print(f"🔥 Starting Zero-shot Evaluation for {MODEL_TYPE} on CK+...")

# ==========================================
# 1. 数据加载 (简单的 ImageFolder)
# ==========================================
def get_ckplus_loader(data_dir):
    # LVLM 不需要复杂的 transform，只需要 Resize 这一步通常 Processor 会做
    # 但为了保险，我们转成 Tensor 前不做 normalize，保留原始 PIL Image
    # 因为 HF 的 Processor 接收 PIL
    dataset = datasets.ImageFolder(data_dir)
    
    # 获取类别索引
    class_to_idx = dataset.class_to_idx
    idx_to_class = {v: k.lower() for k, v in class_to_idx.items()}
    print(f"✅ Classes found: {idx_to_class}")
    
    # 我们这里不需要 DataLoader 的 collate_fn，因为我们要喂 PIL 图片给模型
    # 所以我们手动写个简单的循环，或者用一个返回 (path, label) 的 Dataset
    return dataset, idx_to_class

# ==========================================
# 2. 模型加载器
# ==========================================
def load_model(model_type):
    if model_type == 'instructblip':
        print("⏳ Loading InstructBLIP-7B (Vicuna)...")
        # 使用 4-bit 量化加载以节省显存
        processor = InstructBlipProcessor.from_pretrained("Salesforce/instructblip-vicuna-7b")
        model = InstructBlipForConditionalGeneration.from_pretrained(
            "Salesforce/instructblip-vicuna-7b",
            load_in_4bit=True,
            torch_dtype=torch.float16,
            device_map="auto"
        )
    elif model_type == 'llava':
        print("⏳ Loading LLaVA-1.5-7B...")
        processor = AutoProcessor.from_pretrained("llava-hf/llava-1.5-7b-hf")
        model = LlavaForConditionalGeneration.from_pretrained(
            "llava-hf/llava-1.5-7b-hf",
            load_in_4bit=True,
            torch_dtype=torch.float16,
            device_map="auto"
        )
    else:
        raise ValueError("Unknown model type")
    
    return model, processor

# ==========================================
# 3. 核心：构造 Prompt 和解析输出
# ==========================================
def get_prompt(model_type, emotion_list_str):
    if model_type == 'instructblip':
        # InstructBLIP 喜欢简短的指令
        return f"Question: What is the emotion of the person in the image? Choose from {emotion_list_str}. Answer:"
    elif model_type == 'llava':
        # LLaVA 标准格式
        return f"USER: <image>\nWhat is the facial expression of this person? Output only one word from the following list: {emotion_list_str}.\nASSISTANT:"

def parse_output(output_text, emotion_map):
    """
    非常关键：大模型可能会输出 "I think it is happy"，我们需要提取核心词 "happy"
    """
    output_text = output_text.lower().strip()
    
    # 移除标点
    import string
    output_text = output_text.translate(str.maketrans('', '', string.punctuation))
    
    # 1. 精确匹配
    for emo in emotion_map.keys():
        # 检查 emo 是否作为独立的词出现
        if emo in output_text.split():
            return emotion_map[emo]
    
    # 2. 模糊匹配 (如果没匹配到，看包含关系)
    for emo in emotion_map.keys():
        if emo in output_text:
            return emotion_map[emo]
            
    return -1 # 解析失败 (Hallucination or formatting error)

# ==========================================
# 4. 推理循环
# ==========================================
def evaluate():
    dataset, idx_to_class_map = get_ckplus_loader(DATA_DIR)
    model, processor = load_model(MODEL_TYPE)
    
    # 构造类别字符串 "anger, contempt, ..."
    # 假设 idx_to_class_map 的 value 是 'anger', 'happy' 等
    # 如果数据集文件夹名是 '0', '1', 你需要手动建立一个 int->str 的映射
    
    # 这里的 idx_to_class_map 是 dataset 自动生成的，如果文件夹名就是 emotion name
    emotion_names = list(EMOTION_MAP.keys())
    emotion_list_str = ", ".join(emotion_names)
    
    y_true = []
    y_pred = []
    
    print("🚀 Start Inference...")
    
    # 遍历数据集
    for i in tqdm(range(len(dataset))):
        image, label_idx = dataset[i] # image is PIL, label_idx is int
        
        # 转换标签 (dataset 的 label_idx 可能和 EMOTION_MAP 不对应，需要校准)
        # 假设 dataset.classes 是 ['anger', 'contempt'...] 且按字母排序
        # 为了稳妥，我们用文件夹名反查
        folder_name = dataset.classes[label_idx].lower()
        
        # 如果文件夹名含有 emotion 关键词 (例如 "1_anger")
        true_label_mapped = -1
        for key in EMOTION_MAP:
            if key in folder_name:
                true_label_mapped = EMOTION_MAP[key]
                break
        
        if true_label_mapped == -1:
            continue # 跳过未定义的类别
            
        y_true.append(true_label_mapped)
        
        # 准备输入
        prompt = get_prompt(MODEL_TYPE, emotion_list_str)
        
        if MODEL_TYPE == 'instructblip':
            inputs = processor(images=image, text=prompt, return_tensors="pt").to(DEVICE)
        elif MODEL_TYPE == 'llava':
            inputs = processor(images=image, text=prompt, return_tensors="pt").to(DEVICE)

        # 生成
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=20,
                do_sample=False, # Greedy decoding (deterministic)
                min_length=1
            )
            
        output_text = processor.batch_decode(output_ids, skip_special_tokens=True)[0].strip()
        
        # 解析
        pred_label = parse_output(output_text, EMOTION_MAP)
        
        # 如果解析失败，默认给个错的或者随机，这里我们给 -1
        y_pred.append(pred_label)
        
        # 打印前几个看看效果
        if i < 5:
            print(f"\nExample {i}:")
            print(f"Prompt: {prompt}")
            print(f"Raw Output: {output_text}")
            print(f"Parsed Pred: {pred_label}, True: {true_label_mapped}")

    # 计算指标
    # 过滤掉解析失败的 (-1)
    valid_indices = [k for k, x in enumerate(y_pred) if x != -1]
    y_true_valid = [y_true[k] for k in valid_indices]
    y_pred_valid = [y_pred[k] for k in valid_indices]
    
    acc = accuracy_score(y_true_valid, y_pred_valid)
    print(f"\n\n🏁 Final Results for {MODEL_TYPE} on CK+:")
    print(f"Parsing Success Rate: {len(valid_indices)/len(y_pred):.2%}")
    print(f"Accuracy: {acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true_valid, y_pred_valid, target_names=emotion_names))

if __name__ == '__main__':
    evaluate()